In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

In [2]:
titanic = sns.load_dataset('titanic')

# 1. inspect raw data
print(titanic.info())
print(titanic.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB
None
survived         0
pclass           0
sex              0
age            177
sibsp        

In [3]:
# 2. Drop useless columns
df = titanic.drop(columns=['deck', 'embark_town', 'alive', 'class'])

In [4]:
# 3. Handle missing values
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df = df.dropna(subset=['fare'])

In [5]:
# 4. Create new features (THIS is feature engineering)
df['family_size'] = df['sibsp'] + df['parch'] + 1
df['is_alone'] = (df['family_size'] == 1).astype(int)
df['fare_per_person'] = df['fare'] / df['family_size']

In [6]:
# Extract title from name (powerful feature!)
df['title'] = df['who'].map({'man': 'Mr', 'woman': 'Mrs', 'child': 'Master'})

# Age bins
df['age_group'] = pd.cut(df['age'], bins=[0, 12, 18, 35, 60, 100],
                         labels=['child', 'teen', 'young_adult', 'adult', 'senior'])

In [7]:
# 5. Encode categoricals
# Binary encoding
df['sex_encoded'] = (df['sex'] == 'female').astype(int)

# one-hot encoding for multi-class categoricals
df = pd.get_dummies(df, columns=['embarked', 'age_group'], drop_first=True)

In [8]:
# 6. Select Final Features
feature_cols = [c for c in df.columns if c not in ['survived', 'who', 'adult_male', 'title']]
x = df[feature_cols].select_dtypes(include='number')
y = df['survived']

print(f"Features: {x.shape[1]}, Samples: {x.shape[0]}")
print(x.head())

Features: 9, Samples: 891
   pclass   age  sibsp  parch     fare  family_size  is_alone  \
0       3  22.0      1      0   7.2500            2         0   
1       1  38.0      1      0  71.2833            2         0   
2       3  26.0      0      0   7.9250            1         1   
3       1  35.0      1      0  53.1000            2         0   
4       3  35.0      0      0   8.0500            1         1   

   fare_per_person  sex_encoded  
0          3.62500            0  
1         35.64165            1  
2          7.92500            1  
3         26.55000            1  
4          8.05000            0  
